In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer

print("PyTorch version:", torch.__version__)
print("Cuda available:", torch.cuda.is_available())
print("Cuda Memory Size:", round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2), "GB")


def log_sum_exp(logits_cut_max: torch.Tensor):
    return torch.log(torch.sum(torch.exp(logits_cut_max), dim=-1, keepdim=True))

def log_softmax(logits: torch.Tensor):
    max_val = torch.max(logits, dim=-1, keepdim=True).values
    return (logits - max_val) - log_sum_exp(logits - max_val)

def softmax(logits: torch.Tensor):
    return torch.exp(log_softmax(logits))

def get_ce_loss(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor):
    logps = log_softmax(logits)
    ce_loss = torch.gather(logps, dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
    loss = (ce_loss * mask).sum() / mask.sum().clamp_min(1.0)
    return loss

def get_square_ce_loss(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor):
    logps = log_softmax(logits)
    ce_loss = torch.gather(logps, dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
    
    weight = mask * torch.sqrt(torch.sum(mask, dim=-1)).unsqueeze(-1)
    print(ce_loss.shape, weight.shape)
    loss = (ce_loss * weight).sum() / weight.sum().clamp_min(1.0)
    return loss

def get_logits(model: nn.Module, input_ids: torch.Tensor):
    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits
    return logits


if __name__ == "__main__":
    # 模拟输入
    input_ids = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8]])
    targets = torch.tensor([[2, 3, 4, 5], [6, 7, 8, 9]])
    attention_mask = torch.tensor([[1, 1, 1, 0], [1, 1, 0, 0]], dtype=torch.float)

    # 模拟模型输出
    class DummyModel(nn.Module):
        def __init__(self):
            super().__init__()

        def forward(self, input_ids):
            batch_size, seq_len = input_ids.shape
            vocab_size = 10
            logits = torch.randn(batch_size, seq_len, vocab_size)
            return type('Output', (object,), {'logits': logits})()

    model = DummyModel()
    logits = get_logits(model, input_ids)
    square_ce_loss = get_square_ce_loss(logits, targets, attention_mask)
    ce_loss = get_ce_loss(logits, targets, attention_mask)
    
    # ppl = torch.exp(ce_loss)

    print(f"Cross-Entropy Loss: {ce_loss:.4f}")
    print(f"Square Cross-Entropy Loss: {square_ce_loss:.4f}")
    # print(f"Perplexity: {ppl:.4f}")


PyTorch version: 2.12.0+rocm7.2
Cuda available: True
Cuda Memory Size: 23.96 GB
torch.Size([2, 4]) torch.Size([2, 4])
Cross-Entropy Loss: -3.5479
Square Cross-Entropy Loss: -3.6120


### Lora

In [ ]:
import torch
import torch.nn as nn


class LoraLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, r: int = 4, alpha: float = 1.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.r = r
        self.alpha = alpha
        
        # 原始权重冻结
        self.weight = nn.Linear(in_features, out_features)
        self.weight.weight.requires_grad = False
        
        # LoRA 权重
        self.A = nn.Linear(in_features, r)
        self.B = nn.Linear(r, out_features)
        nn.init.normal_(self.A.weight, std=1e-5)
        nn.init.zeros_(self.B.weight)
    
    
    def forward(self, x):
        original_out = self.weight(x) # (batch_size, out_features)
        lora_out = self.B(self.A(x)) * (self.alpha / self.r) # (batch_size, out_features)
        return original_out + lora_out

# RLs

### DPO

$r_c = log(pi_theta_c) - log(pi_old_c)$

r_w = log(pi_theta_w) - log(pi_old_w)

L_{dpo} = -E[log_sigmoid(beta * (r_c - r_w))] 

In [1]:
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Optional, Tuple

# -------------------------
# 手动稳定函数：logsumexp / log_softmax / softplus / log_sigmoid
# -------------------------

def logsumexp(x: torch.Tensor, dim: int = -1, keepdim: bool = False) -> torch.Tensor:
    m = x.max(dim=dim, keepdim=True).values
    y = m + torch.log(torch.sum(torch.exp(x - m), dim=dim, keepdim=True))
    return y if keepdim else y.squeeze(dim)

def log_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    return x - logsumexp(x, dim=dim, keepdim=True)

def softplus(x: torch.Tensor) -> torch.Tensor:
    return torch.clamp(x, min=0) + torch.log1p(torch.exp(-torch.abs(x)))

def log_sigmoid(x: torch.Tensor) -> torch.Tensor:
    return -softplus(-x)

# -------------------------
# 手动 token logp（cLM）
# -------------------------

def per_token_logps_from_logits(
    logits: torch.Tensor,          # [B, T, V]
    input_ids: torch.Tensor,       # [B, T]
) -> torch.Tensor:
    """
    logits[:, t] 预测 input_ids[:, t+1]
    返回 [B, T-1]
    """
    logits = logits[:, :-1, :].float()           # FP32 更稳
    targets = input_ids[:, 1:]
    lprobs = log_softmax(logits, dim=-1)
    tok = torch.gather(lprobs, dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
    return tok

def forward_token_logps(
    model: nn.Module,
    input_ids: torch.Tensor,
    attention_mask: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    return per_token_logps_from_logits(out.logits, input_ids)  # [B, T-1]

def sequence_logp(token_logps: torch.Tensor, token_mask: torch.Tensor) -> torch.Tensor:
    """
    token_logps: [B, T-1]
    token_mask : [B, T-1] 1=计入（通常只计 completion）
    """
    return (token_logps * token_mask.float()).sum(dim=1)  # [B]

# -------------------------
# DPO batch
# -------------------------

@dataclass
class DPOBatch:
    chosen_input_ids: torch.Tensor        # [B, T]
    chosen_attn_mask: torch.Tensor        # [B, T]
    rejected_input_ids: torch.Tensor      # [B, T]
    rejected_attn_mask: torch.Tensor      # [B, T]
    chosen_completion_mask: torch.Tensor  # [B, T-1]
    rejected_completion_mask: torch.Tensor# [B, T-1]

# -------------------------
# 手动 DPO loss
# -------------------------

def dpo_loss(
    policy_model: nn.Module,
    ref_model: nn.Module,
    batch: DPOBatch,
    beta: float = 0.1,
) -> torch.Tensor:
    """
    DPO:
      diff = (logπ(y+) - logπ(y-)) - (logref(y+) - logref(y-))
      loss = -log sigmoid(beta * diff)
    """
    # policy logps (grad)
    pi_c_tok = forward_token_logps(policy_model, batch.chosen_input_ids, batch.chosen_attn_mask)    # [B, T-1]
    pi_r_tok = forward_token_logps(policy_model, batch.rejected_input_ids, batch.rejected_attn_mask)

    # ref logps (no grad)
    with torch.no_grad():
        rf_c_tok = forward_token_logps(ref_model, batch.chosen_input_ids, batch.chosen_attn_mask)
        rf_r_tok = forward_token_logps(ref_model, batch.rejected_input_ids, batch.rejected_attn_mask)

    # sequence logp over completion only
    pi_c = sequence_logp(pi_c_tok, batch.chosen_completion_mask)     # [B]
    pi_r = sequence_logp(pi_r_tok, batch.rejected_completion_mask)   # [B]
    rf_c = sequence_logp(rf_c_tok, batch.chosen_completion_mask)     # [B]
    rf_r = sequence_logp(rf_r_tok, batch.rejected_completion_mask)   # [B]

    diff = (pi_c - pi_r) - (rf_c - rf_r)       # [B]
    loss = (-log_sigmoid(beta * diff)).mean()  # scalar
    return loss

# -------------------------
# train step
# -------------------------

def dpo_train_step(
    policy_model: nn.Module,
    ref_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    batch: DPOBatch,
    beta: float,
    grad_accum_steps: int = 1,
    step_idx: int = 1,
) -> float:
    policy_model.train()

    loss = dpo_loss(policy_model, ref_model, batch, beta=beta)
    loss = loss / float(grad_accum_steps)
    loss.backward()

    if step_idx % grad_accum_steps == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    return float(loss.detach().cpu().item() * grad_accum_steps)

# -------------------------
# completion_mask 怎么做（提示）
# -------------------------
def make_completion_mask_from_prompt_len(prompt_len: torch.Tensor, T: int, device) -> torch.Tensor:
    """
    prompt_len: [B]（例如 attention_mask.sum(dim=1)）
    返回 [B, T-1]，1 表示对应预测位置属于 completion（k>=prompt_len）
    token_logps 的 index i 对应预测 token k=i+1
    """
    B = prompt_len.size(0)
    k = torch.arange(1, T, device=device).unsqueeze(0).expand(B, T-1)  # k=1..T-1
    return (k >= prompt_len.unsqueeze(1)).int()


### GRPO

$\mathcal{L}_{GRPO} = -\mathbb{E}_{q \sim P(Q), \{o_i\}_{i=1}^G \sim \pi_{\theta_{old}}} \left[ \frac{1}{G} \sum_{i=1}^G \left( \min\left(\rho_i \hat{A}_i, \text{clip}(\rho_i, 1-\epsilon, 1+\epsilon) \hat{A}_i\right) - \beta \mathbb{D}_{KL}(\pi_\theta \| \pi_{ref}) \right) \right]$




In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------
# 1. 工具：取 logprobs + mask
# ----------------------
def get_logprob_and_mask(model, input_ids, attention_mask, prompt_len):
    bs, seq_len = input_ids.shape
    out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    logits = out.logits                     # [bs, seq_len, vocab]

    # shift
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()

    logp_all = F.log_softmax(shift_logits, dim=-1)
    logprobs = logp_all.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)  # [bs, seq_len-1]

    # 构造 mask：对应 logits 的位置，长度 seq_len-1
    mask = attention_mask[:, 1:].float()    # 原 mask 右移，标记标签有效性
    loss_mask = mask.clone()
    loss_mask[:, :prompt_len-1] = 0.0      # 屏蔽 prompt 部分对应的 logits

    return logprobs, loss_mask   # 二者长度均为 seq_len-1

# ----------------------
# 2. 完整版 GRPO Loss 计算
# ----------------------
def grpo_loss(
    policy_model: nn.Module,    # 当前策略模型
    ref_model: nn.Module,       # 参考/旧模型
    input_ids: torch.LongTensor,
    attention_mask: torch.LongTensor,
    prompt_len: int,
    rewards: torch.Tensor,      # [bs,] 每条样本全局奖励
    eps_clip: float = 0.2,
    beta: float = 0.01          # KL 散度系数
):
    """
    标准 GRPO: PPO-Clip + KL 正则 + mask 屏蔽无效位置
    """
    device = input_ids.device

    # ========== 1. 取当前策略 logprob & mask ==========
    curr_logp, loss_mask = get_logprob_and_mask(
        policy_model, input_ids, attention_mask, prompt_len
    )

    # ========== 2. 取参考模型 old logprob & KL ==========
    with torch.no_grad():
        ref_logp, _ = get_logprob_and_mask(
            ref_model, input_ids, attention_mask, prompt_len
        )

    # ========== 4. 计算概率比率 r = exp(logp - old_logp) ==========
    ratio = torch.exp(curr_logp - ref_logp)

    # ========== 5. 构造优势 Advantage (GRPO 用全局奖励) ==========
    # 广播奖励到每个token
    valid_cnt = loss_mask.sum(dim=1)  # [bs] 每条有效token数
    adv = rewards.unsqueeze(1).repeat(1, input_ids.size(1))  # [bs, seq_len]

    # 优势归一化
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)

    # ========== 6. PPO Clip Loss ==========
    surr1 = ratio * adv
    surr2 = torch.clamp(ratio, 1-eps_clip, 1+eps_clip) * adv
    ppo_loss = torch.min(surr1, surr2)

    # ========== 7. KL 散度正则 ==========
    kl = curr_logp - ref_logp
    total_loss = -torch.mean(ppo_loss - beta * kl)
    total_loss = (total_loss * loss_mask).sum() / loss_mask.sum().clamp_min(1.0)  # 平均到每个有效token

    return total_loss, ppo_loss.item(), kl.item()

### PPO

In [ ]:
import torch.nn as nn


class PPO(nn.Module):
    def __init__(self):
        super().__init__()
        self.lam = 0.99
        self.gam = 0.95
    
    def cal_adv(self, rewards, values):
        gae = 0.0
        
        
        
        
        
        
        
        
        

IndexError: index 31 is out of bounds for dimension 0 with size 16

## SGD 

In [8]:
import numpy as np

np.random.seed(42)  # 保证可复现



# ============ SwiGLU 激活函数 ============
def silu(x):
    """Swish/SiLU: x * sigmoid(x)"""
    return x * (1 / (1 + np.exp(-x)))


def silu_backward(x):
    """SiLU 的导数: sigmoid(x) + x * sigmoid(x) * (1 - sigmoid(x))"""
    sig = 1 / (1 + np.exp(-x))
    return sig + x * sig * (1 - sig)



def adam_mlp2(X, y, hidden_dim=64, lr=0.01, epochs=100, batch_size=32,
              beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
    """
    Adam 优化的 2 层 MLP（MSE 回归）
    
    架构: X -> Linear -> ReLU -> Linear -> y_pred
    Loss: MSE = mean((y_pred - y)^2)
    """
    
    n_samples, n_features = X.shape
    
    # ============ He 初始化（适合 ReLU）============
    W1 = np.random.randn(n_features, hidden_dim) * np.sqrt(2 / n_features)
    b1 = np.zeros(hidden_dim)
    W2 = np.random.randn(hidden_dim, 1) * np.sqrt(2 / hidden_dim)
    b2 = np.zeros(1)
    
    # ============ Adam 一阶矩 m 和二阶矩 v ============
    m_W1, v_W1 = np.zeros_like(W1), np.zeros_like(W1)
    m_b1, v_b1 = np.zeros_like(b1), np.zeros_like(b1)
    m_W2, v_W2 = np.zeros_like(W2), np.zeros_like(W2)
    m_b2, v_b2 = np.zeros_like(b2), np.zeros_like(b2)
    t = 0  # 时间步，用于 bias correction
    
    losses = []
    
    for epoch in range(epochs):
        idx = np.random.permutation(n_samples)
        epoch_loss = 0.0
        n_batches = 0
        
        for i in range(0, n_samples, batch_size):
            t += 1
            Xb = X[idx[i:i+batch_size]]
            yb = y[idx[i:i+batch_size]].reshape(-1, 1)
            
            # ==================== Forward ====================
            # z1 = Xb @ W1 + b1       shape: (batch, hidden)
            # h1 = ReLU(z1)           shape: (batch, hidden)
            # yp = h1 @ W2 + b2       shape: (batch, 1)
            # loss = mean((yp - yb)^2)
            
            z1 = Xb @ W1 + b1
            h1 = np.maximum(0, z1)  # ReLU 
            yp = h1 @ W2 + b2
            
            batch_loss = np.mean((yp - yb) ** 2)
            print(f"Current batch loss: {batch_loss:.4f}")
            epoch_loss += batch_loss
            n_batches += 1
            
            # ==================== Backward ====================
            """
            推导过程:
            
            Loss = (1/n) * Σ(yp - y)^2
            
            【Layer 2】
            ∂L/∂yp = (2/n) * (yp - y)              shape: (batch, 1)
            
            ∂L/∂W2 = ∂L/∂yp · ∂yp/∂W2 
                   = h1.T @ ∂L/∂yp                  shape: (hidden, 1)
            
            ∂L/∂b2 = ∂L/∂yp · ∂yp/∂b2 
                   = sum(∂L/∂yp, axis=0)            shape: (1,)
            
            【Layer 1】
            ∂L/∂h1 = ∂L/∂yp · ∂yp/∂h1 
                   = ∂L/∂yp @ W2.T                  shape: (batch, hidden)
            
            ∂L/∂z1 = ∂L/∂h1 · ∂h1/∂z1 
                   = ∂L/∂h1 * ReLU'(z1) 
                   = ∂L/∂h1 * (z1 > 0)              shape: (batch, hidden)
            
            ∂L/∂W1 = ∂L/∂z1 · ∂z1/∂W1 
                   = Xb.T @ ∂L/∂z1                  shape: (features, hidden)
            
            ∂L/∂b1 = sum(∂L/∂z1, axis=0)            shape: (hidden,)
            """
            
            # Layer 2 梯度
            loss_grad = 2 / len(Xb) * (yp - yb)  # ∂L/∂yp
            
            dW2 = h1.T @ loss_grad               # ∂L/∂W2
            db2 = np.sum(loss_grad, axis=0)      # ∂L/∂b2
            
            # Layer 1 梯度
            dz1 = (loss_grad @ W2.T) * (z1 > 0)  # ∂L/∂z1 = ∂L/∂h1 * ReLU'
            dW1 = Xb.T @ dz1                     # ∂L/∂W1
            db1 = np.sum(dz1, axis=0)            # ∂L/∂b1
            
            # ==================== Adam 更新 ====================
            """
            Adam 更新规则:
            m_t = β1 * m_{t-1} + (1-β1) * g_t        一阶矩（动量）
            v_t = β2 * v_{t-1} + (1-β2) * g_t^2      二阶矩（RMSprop）
            m_hat = m_t / (1 - β1^t)                 bias correction
            v_hat = v_t / (1 - β2^t)                 bias correction
            θ = θ - lr * m_hat / (√v_hat + ε)        参数更新
            
            Weight Decay (AdamW 风格):
            θ = θ - lr * (m_hat / (√v_hat + ε) + λ * θ)
            """
            
            for param, grad, m_p, v_p in [
                (W1, dW1, m_W1, v_W1), (b1, db1, m_b1, v_b1),
                (W2, dW2, m_W2, v_W2), (b2, db2, m_b2, v_b2),
            ]:
                # 更新一阶矩和二阶矩
                m_p[:] = beta1 * m_p + (1 - beta1) * grad
                v_p[:] = beta2 * v_p + (1 - beta2) * (grad ** 2)
                
                # Bias correction（修正初始零偏置）
                m_hat = m_p / (1 - beta1 ** t)
                v_hat = v_p / (1 - beta2 ** t)
                
                # Weight decay 只用于权重，不用 bias
                wd = weight_decay if param.ndim == 2 else 0
                
                # 参数更新
                param -= lr * (m_hat / (np.sqrt(v_hat) + eps) + wd * param)
        
        losses.append(epoch_loss / n_batches)
    
    return W1, b1, W2, b2, losses


def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


# ============ 测试代码 ============
X_train = np.random.randn(100, 10)
y_train = np.random.randn(100)
X_test = np.random.randn(20, 10)
y_test = np.random.randn(20)

# 对比 ReLU MLP 和 SwiGLU MLP
print("=" * 50)
print("ReLU MLP:")
W1, b1, W2, b2, losses_relu = adam_mlp2(X_train, y_train, hidden_dim=32, lr=0.01, epochs=50)
z1 = X_test @ W1 + b1
y_pred_relu = (np.maximum(0, z1) @ W2 + b2).ravel()
print(f"Test MSE = {mean_squared_error(y_test, y_pred_relu):.4f}")
print(f"Train loss: {losses_relu[0]:.4f} -> {losses_relu[-1]:.4f}")

# print("\n" + "=" * 50)
# print("SwiGLU MLP:")
# params_swiglu, losses_swiglu = adam_mlp_swiglu(X_train, y_train, hidden_dim=32, lr=0.01, epochs=50)
# z_gate = X_test @ params_swiglu['W_gate'] + params_swiglu['b_gate']
# z_up = X_test @ params_swiglu['W_up'] + params_swiglu['b_up']
# h_test = silu(z_gate) * z_up
# y_pred_swiglu = (h_test @ params_swiglu['W_down'] + params_swiglu['b_down']).ravel()
# print(f"Test MSE = {mean_squared_error(y_test, y_pred_swiglu):.4f}")
# print(f"Train loss: {losses_swiglu[0]:.4f} -> {losses_swiglu[-1]:.4f}")

ReLU MLP:
Current batch loss: 2.5803
Current batch loss: 1.1264
Current batch loss: 1.6125
Current batch loss: 1.0386
Current batch loss: 0.7152
Current batch loss: 1.3922
Current batch loss: 1.2485
Current batch loss: 1.3668
Current batch loss: 1.0369
Current batch loss: 0.6675
Current batch loss: 1.1696
Current batch loss: 0.9776
Current batch loss: 0.8199
Current batch loss: 0.5776
Current batch loss: 1.0259
Current batch loss: 1.0887
Current batch loss: 0.6482
Current batch loss: 0.7385
Current batch loss: 0.8663
Current batch loss: 0.5920
Current batch loss: 0.7028
Current batch loss: 0.6890
Current batch loss: 0.7394
Current batch loss: 0.5470
Current batch loss: 0.6228
Current batch loss: 0.6653
Current batch loss: 0.7394
Current batch loss: 0.3973
Current batch loss: 0.7110
Current batch loss: 0.6308
Current batch loss: 0.5929
Current batch loss: 0.5888
Current batch loss: 0.6161
Current batch loss: 0.6292
Current batch loss: 0.5706
Current batch loss: 0.7171
Current batch loss

## Traditional RLs

1. Q-Learning：离线策略，Q(s,a) += lr*(r+γmaxQ'-Q)
2. DQN：经验回放 + 目标网络 + MSE 损失
3. REINFORCE：回合更新，折扣回报归一化，loss=-Σlogπ*R
4. SAC：双 Q + 最大熵 + 重参数化 + 软更新

In [ ]:
import numpy as np
import gymnasium as gym

# 超参
lr = 0.1
gamma = 0.9
epsilon = 0.1
episodes = 500

env = gym.make("CliffWalking-v1")
n_states = env.observation_space.n
n_actions = env.action_space.n

# Q表
Q = np.zeros((n_states, n_actions))

for _ in range(episodes):
    state, _ = env.reset()
    done = False
    while not done:
        # epsilon贪心
        if np.random.uniform() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q[state])
        
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        # Q学习更新公式
        print("cur Q Table: ", Q)
        print(f"cur reward: {reward}, cur_state: {state}, action: {action}, next_state: {next_state}")
        print("#" * 100)
        Q[state, action] = Q[state, action] + lr * (
            reward + gamma * np.max(Q[next_state]) - Q[state, action]
        )
        state = next_state

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64), nn.ReLU(),
            nn.Linear(64, action_dim)
        )
    def forward(self, x):
        return self.fc(x)

# 超参
gamma = 0.99
lr = 1e-3
epsilon = 0.1
buffer_size = 10000
batch_size = 32
target_update = 10

env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

net = DQN(state_dim, action_dim)
target_net = DQN(state_dim, action_dim)
target_net.load_state_dict(net.state_dict())
optimizer = optim.Adam(net.parameters(), lr=lr)
buffer = deque(maxlen=buffer_size)

for episode in range(500):
    state, _ = env.reset()
    total_reward = 0
    while True:
        # 选动作
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = net(torch.FloatTensor(state)).argmax().item()
        
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        buffer.append((state, action, reward, next_state, done))
        
        total_reward += reward
        state = next_state
        
        # 训练
        if len(buffer) > batch_size:
            batch = random.sample(buffer, batch_size)
            s, a, r, s_, done = zip(*batch)
            s = torch.FloatTensor(s)
            a = torch.LongTensor(a).unsqueeze(1)
            r = torch.FloatTensor(r).unsqueeze(1)
            s_ = torch.FloatTensor(s_)
            done = torch.FloatTensor(done).unsqueeze(1)
            
            q = net(s).gather(1, a)
            max_q = target_net(s_).max(1, keepdim=True)[0]
            target_q = r + gamma * max_q * (1 - done)
            
            loss = nn.MSELoss()(q, target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Episode {episode}, Total Reward: {total_reward}")
        print(f"Buffer Size: {len(buffer)}, Loss: {loss.item() if 'loss' in locals() else 'N/A'}")
        print(done)
        if not isinstance(done, bool):
            break
    
    # 更新目标网络
    if episode % target_update == 0:
        target_net.load_state_dict(net.state_dict())

In [1]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim

class Policy(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64), nn.ReLU(),
            nn.Linear(64, action_dim), nn.Softmax(dim=-1)
        )
    def forward(self, x):
        return self.fc(x)

gamma = 0.99
lr = 1e-3

env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

policy = Policy(state_dim, action_dim)
optimizer = optim.Adam(policy.parameters(), lr=lr)

for episode in range(1000):
    state, _ = env.reset()
    log_probs = []
    rewards = []
    
    while True:
        prob = policy(torch.FloatTensor(state))
        action = torch.multinomial(prob, 1).item()
        log_prob = torch.log(prob[action])
        
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        log_probs.append(log_prob)
        rewards.append(reward)
        state = next_state
        
        if done:
            # 计算折扣回报
            returns = []
            R = 0
            for r in reversed(rewards):
                R = r + gamma * R
                returns.insert(0, R)
            returns = torch.tensor(returns)
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
            
            # 策略梯度损失
            loss = -torch.sum(torch.stack(log_probs) * returns)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            break

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import random
from collections import deque

# 双Q网络
class QNet(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()
        self.q1 = nn.Sequential(nn.Linear(s_dim+a_dim,64),nn.ReLU(),nn.Linear(64,1))
        self.q2 = nn.Sequential(nn.Linear(s_dim+a_dim,64),nn.ReLU(),nn.Linear(64,1))
    def forward(self, s, a):
        x = torch.cat([s,a],1)
        return self.q1(x), self.q2(x)

# 策略网络
class Actor(nn.Module):
    def __init__(self, s_dim, a_dim, max_a):
        super().__init__()
        self.max_a = max_a
        self.fc = nn.Sequential(nn.Linear(s_dim,64),nn.ReLU())
        self.mean = nn.Linear(64, a_dim)
        self.log_std = nn.Linear(64, a_dim)
    def forward(self, s):
        x = self.fc(s)
        mean = self.mean(x)
        log_std = torch.clamp(self.log_std(x),-20,2)
        return mean, log_std
    def sample(self, s):
        mean, log_std = self.forward(s)
        dist = Normal(mean, log_std.exp())
        x = dist.rsample()
        y = torch.tanh(x)
        log_prob = dist.log_prob(x) - torch.log(1-y.pow(2)+1e-6)
        return y*self.max_a, log_prob.sum(1,keepdim=True)

# 训练
gamma=0.99;tau=0.005;lr=3e-4
env = gym.make("Pendulum-v1")
s_dim = env.observation_space.shape[0]
a_dim = env.action_space.shape[0]
max_a = env.action_space.high[0]

q_net = QNet(s_dim,a_dim)
q_target = QNet(s_dim,a_dim)
q_target.load_state_dict(q_net.state_dict())
actor = Actor(s_dim,a_dim,max_a)

q_opt = optim.Adam(q_net.parameters(),lr=lr)
a_opt = optim.Adam(actor.parameters(),lr=lr)
buffer = deque(maxlen=10000)

for episode in range(1000):
    s,_ = env.reset()
    while True:
        a,_ = actor.sample(torch.FloatTensor(s).unsqueeze(0))
        a = a.detach().numpy()[0]
        s_,r,terminated,truncated,_ = env.step(a)
        done = terminated or truncated
        buffer.append((s,a,r,s_,done))
        s = s_
        
        if len(buffer)>32:
            batch = random.sample(buffer,32)
            s,a,r,s_,done = zip(*batch)
            s = torch.FloatTensor(s)
            a = torch.FloatTensor(a)
            r = torch.FloatTensor(r).unsqueeze(1)
            s_ = torch.FloatTensor(s_)
            done = torch.FloatTensor(done).unsqueeze(1)
            
            # 更新Q
            a_,log_p = actor.sample(s_)
            q1_t,q2_t = q_target(s_,a_)
            target_q = r+gamma*(1-done)*(torch.min(q1_t,q2_t)-log_p)
            q1,q2 = q_net(s,a)
            q_loss = nn.MSELoss()(q1,target_q)+nn.MSELoss()(q2,target_q)
            q_opt.zero_grad()
            q_loss.backward()
            q_opt.step()
            
            # 更新Actor
            a_pred,log_p = actor.sample(s)
            q1_min,q2_min = q_net(s,a_pred)
            a_loss = (log_p - torch.min(q1_min,q2_min)).mean()
            a_opt.zero_grad()
            a_loss.backward()
            a_opt.step()
            
            # 软更新
            for p,tp in zip(q_net.parameters(),q_target.parameters()):
                tp.data.copy_(tau*p.data+(1-tau)*tp.data)
        if done:
            break

In [4]:
import torch
import torch.nn as nn


class fn(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, dropout: float = 0.1):
        super().__init__()
        self.up_proj = nn.Linear(input_dim, output_dim * 4)
        self.down_proj = nn.Linear(output_dim * 4, output_dim)
        self.gated_act = nn.SiLU()
        self.gated_proj = nn.Linear(input_dim, output_dim * 4)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        up = self.up_proj(x)
        gate = self.gated_proj(x)
        gated_up = up * self.gated_act(gate)
        dropped = self.dropout(gated_up)
        output = self.down_proj(dropped)
        return output
    
class moefn(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, num_experts: int = 4, top_k: int = 2, dropout: float = 0.1):
        super().__init__()
        self.experts = nn.ModuleList([fn(input_dim, output_dim, dropout) for _ in range(num_experts)])
        self.router = nn.Linear(input_dim, num_experts)
        self.top_k = top_k
        self.aux_loss_coef = 0.01
    
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        router_logits = self.router(x)
        router_probs = torch.softmax(router_logits, dim=-1) # [batch_size, seq_len, num_experts]
        topk_probs, topk_indices = torch.topk(router_probs, self.top_k, dim=-1) # (bsz, seq_len, top_k)
        expert_outputs = torch.stack([self.experts[i](x) for i in range(len(self.experts))], dim=2) # (batch_size, seq_len, num_experts, output_dim)
        # index: (batch_size, seq_len, num_experts, 1) -> (batch_size, seq_len, num_experts, output_dim) -> (batch_size, seq_len, top_k, output_dim)
        topk_expert_outputs = torch.gather(expert_outputs, dim=2, index=topk_indices.unsqueeze(-1).expand(-1, -1, -1, expert_outputs.shape[-1]))
        print(topk_expert_outputs.shape)
        topk_expert_outputs = topk_expert_outputs * topk_probs.unsqueeze(-1) # (batch_size, seq_len, top_k, output_dim)
        output = topk_expert_outputs.sum(dim=2) # (batch_size, seq_len, output_dim)
        aux_loss = self.aux_loss_coef * (router_probs.mean(dim=1) * router_probs.mean(dim=1)).sum() # encourage balanced routing
        return output, aux_loss


moe = moefn(input_dim=16, output_dim=32, num_experts=4, top_k=2)
x = torch.randn(8, 10, 16)
output, aux_loss = moe(x)

print(f"Output shape: {output.shape}, Aux loss: {aux_loss.item():.4f}")

torch.Size([8, 10, 2, 32])
Output shape: torch.Size([8, 10, 32]), Aux loss: 0.0205


### MLA

In [ ]:
import torch
import torch.nn as nn
import math

# ==================== 核心：标准RoPE实现（无修改，官方版） ====================
def precompute_freqs_cis(dim: int, seq_len: int, theta: float = 10000.0, device: torch.device = None):
    """预计算RoPE复数频率张量"""
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, dtype=torch.float32, device=device) / dim))
    t = torch.arange(seq_len, dtype=torch.float32, device=device)
    freqs = torch.outer(t, freqs)
    # 复数表示：cosθ + i*sinθ
    # freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    cos =  torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)
    sin =  torch.cat([torch.sin(freqs), -torch.sin(freqs)], dim=-1)
    freqs_cis = torch.complex(cos, sin)
    return freqs_cis

def apply_rope(x: torch.Tensor, freqs_cis: torch.Tensor):
    """
    为张量应用RoPE旋转
    x: [bsz, seq_len, n_heads, head_dim]
    freqs_cis: [seq_len, head_dim//2]
    """
    # 转换为复数
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    # 广播维度适配
    freqs_cis = freqs_cis.view(1, x.shape[1], 1, -1)
    # 旋转
    x_rotated = torch.view_as_real(x_complex * freqs_cis).flatten(3)
    return x_rotated.type_as(x)

# ==================== 修正版：DeepSeek MLA + 解耦RoPE注入 ====================
class DeepSeekMLA(nn.Module):
    def __init__(
        self,
        hidden_size: int = 2048,        # 模型隐藏层维度
        num_heads: int = 16,             # 注意力头数
        kv_lora_rank: int = 512,         # KV低秩压缩维度（核心MLA参数）
        qk_rope_head_dim: int = 64,      # RoPE位置分支维度（解耦核心）
        rope_theta: float = 10000.0,     # RoPE基础频率
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.kv_lora_rank = kv_lora_rank
        self.qk_rope_head_dim = qk_rope_head_dim
        # 解耦拆分：内容分支(nope) + 位置分支(rope)
        self.qk_nope_head_dim = self.head_dim - self.qk_rope_head_dim
        

        # ==================== MLA 核心投影层（官方标准配置） ====================
        # 1. Q 低秩分解 (Down -> Up)
        self.q_down = nn.Linear(hidden_size, hidden_size, bias=False)
        self.q_up = nn.Linear(hidden_size, num_heads * self.head_dim, bias=False)

        # 2. KV 低秩压缩投影（仅压缩内容分支，位置分支不压缩）
        self.kv_down = nn.Linear(hidden_size, kv_lora_rank, bias=False)
        self.kv_up = nn.Linear(
            kv_lora_rank,
            num_heads * self.qk_nope_head_dim + hidden_size,  # K_nope + V
            bias=False
        )

        # 3. K_rope 独立投影（RoPE专用，不经过低秩压缩！MLA核心设计）
        self.k_rope = nn.Linear(hidden_size, self.qk_rope_head_dim, bias=False)

        # 4. 输出投影
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

        # RoPE频率缓存
        self.rope_theta = rope_theta
        self.freqs_cis = None

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None):
        """
        x: [bsz, seq_len, hidden_size]
        return: [bsz, seq_len, hidden_size]
        """
        bsz, seq_len, _ = x.shape
        device = x.device
        scale = math.sqrt(self.head_dim)

        # ==================== 1. 动态预计算RoPE频率 ====================
        if self.freqs_cis is None or self.freqs_cis.size(0) < seq_len:
            self.freqs_cis = precompute_freqs_cis(
                self.qk_rope_head_dim, seq_len, self.rope_theta, device
            )

        # ==================== 2. Q 投影 + 解耦拆分（nope / rope） ====================
        q = self.q_up(self.q_down(x))  # [bsz, seq_len, num_heads * head_dim]
        q = q.view(bsz, seq_len, self.num_heads, self.head_dim)
        # 解耦：内容分支(不旋转) + 位置分支(仅RoPE)
        q_nope, q_rope = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1)

        # ==================== 3. RoPE 注入：仅作用于位置分支 ====================
        q_rope = apply_rope(q_rope, self.freqs_cis)

        # ==================== 4. KV 低秩解压缩 + K_rope 独立投影 ====================
        # KV低秩压缩 -> 解压缩出 K_nope + V
        kv = self.kv_up(self.kv_down(x))
        k_nope = kv[..., :self.num_heads * self.qk_nope_head_dim].view(
            bsz, seq_len, self.num_heads, self.qk_nope_head_dim
        )
        v = kv[..., self.num_heads * self.qk_nope_head_dim:].view(
            bsz, seq_len, self.num_heads, self.head_dim
        )

        # K_rope：独立投影 + RoPE注入（不经过低秩压缩！）
        k_rope = self.k_rope(x).view(bsz, seq_len, 1, self.qk_rope_head_dim)
        k_rope = k_rope.repeat(1, 1, self.num_heads, 1)  # 共享K_rope到所有头
        k_rope = apply_rope(k_rope, self.freqs_cis)

        # ==================== 5. 维度转置（适配注意力计算） ====================
        # [bsz, num_heads, seq_len, dim]
        q_nope = q_nope.transpose(1, 2)
        q_rope = q_rope.transpose(1, 2)
        k_nope = k_nope.transpose(1, 2)
        k_rope = k_rope.transpose(1, 2)
        v = v.transpose(1, 2)

        # ==================== 6. MLA 解耦注意力计算（官方核心） ====================
        # 内容分支得分 + 位置分支得分 = 总得分
        attn_nope = torch.matmul(q_nope, k_nope.transpose(-2, -1))
        attn_rope = torch.matmul(q_rope, k_rope.transpose(-2, -1))
        attn_scores = (attn_nope + attn_rope) / scale

        # 掩码
        if attention_mask is not None:
            attn_scores = attn_scores.masked_fill(attention_mask == 0, -torch.inf)

        # softmax + 加权求和
        attn_probs = torch.softmax(attn_scores, dim=-1)
        attn_output = torch.matmul(attn_probs, v)

        # ==================== 7. 输出拼接 + 投影 ====================
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(bsz, seq_len, self.hidden_size)
        output = self.o_proj(attn_output)

        return output

# ==================== 测试代码（验证可运行） ====================
if __name__ == "__main__":
    # 模拟DeepSeek-V2小配置
    model = DeepSeekMLA(
        hidden_size=2048,
        num_heads=16,
        kv_lora_rank=512,
        qk_rope_head_dim=64
    )
    # 随机输入
    x = torch.randn(2, 128, 2048)  # [bsz=2, seq_len=128, hidden_size=2048]
    # 前向传播
    out = model(x)
    print(f"输入形状: {x.shape}")
    print(f"输出形状: {out.shape}")
    print("✅ 代码运行成功！MLA + RoPE注入逻辑完全正确")

输入形状: torch.Size([2, 128, 2048])
输出形状: torch.Size([2, 128, 2048])
✅ 代码运行成功！MLA + RoPE注入逻辑完全正确


In [ ]:
# Top-p

import torch


def get_topp(logits, p, temperature=1.0):
    logits = logits / temperature
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
    mask = cumulative_probs > p
    # mask[..., 1:] = mask[..., :-1].clone()
    mask = torch.roll(mask, shifts=1, dims=-1)
    mask[..., 0] = False
    sorted_logits[mask] = float('-inf')
    return sorted_logits.scatter(1, sorted_indices, sorted_logits)
    

def get_topk(logits, k):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    topk_logits = sorted_logits[:, :k]
    topk_indices = sorted_indices[:, :k]
    return topk_logits, topk_indices


def beam_search(logits, beam_width):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    topk_logits = sorted_logits[:, :beam_width]
    topk_indices = sorted_indices[:, :beam_width]
    return topk_logits, topk_indices



logits = torch.tensor([[0.1, 0.2, 0.3, 0.4]])
p = 0.6
filtered_logits = get_topp(logits, p)
print(filtered_logits)

tensor([[  -inf, 0.2000, 0.3000, 0.4000]])


In [1]:
import torch

# a = torch.tensor([1, 0,0,0,0,0,0,0])
a = torch.ones((8,))
a = (torch.rand(a.shape) > 0.9) * a
print(a)
adv = (a - torch.mean(a, dim=-1, keepdim=True)) / (torch.std(a, dim=-1, keepdim=True) + 1e-8)



tensor([0., 0., 0., 0., 0., 0., 0., 0.])


In [62]:
import numpy as np

x = np.random.rand(1000000)
y = np.random.rand(1000000)

val = (x**2 + y**2) <= 1


print(4 * val.sum() / len(x))


3.143284


In [3]:
import torch

a = torch.randn(2, 4, 8) # bsz, seq_len, hidden_size

def safe_softmax(logits):
    max_val = torch.max(logits, dim=-1, keepdim=True).values
    logits = torch.exp(logits - max_val)
    return logits / torch.sum(logits, dim=-1, keepdim=True)


def ce_loss(logits, targets):
    delta = logits - torch.logsumexp(logits, dim=-1, keepdim=True)
    print(targets.unsqueeze(-1).shape, delta.shape)
    return -torch.mean(torch.gather(delta, dim=-1, index=targets.unsqueeze(-1)).squeeze(-1))


logits = torch.randn(2, 4, 8)
targets = torch.randint(0, 8, (2, 4))
print(ce_loss(logits, targets))
# probs = safe_softmax(logits)
# print(probs)
    



torch.Size([2, 4, 1]) torch.Size([2, 4, 8])
tensor(2.2742)


In [5]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


def get_mean(points, k=4, iter=1000):
    center_points = points[np.random.randint(0, len(points), (k,))]
    print(center_points)
    for iter in range(iter):
        clusters = defaultdict(list)
        for point in points:
            dist = np.linalg.norm(point - center_points, axis=-1)
            min_idx = np.argmin(dist, axis=-1)
            clusters[min_idx].append(point)
        for i in range(k):
            center_points[i] = np.mean(clusters[i], axis=0)
    
    cluster_idxs = np.zeros(len(points), dtype=int)
    for idx, point in enumerate(points):
        dist = np.linalg.norm(point - center_points, axis=-1)
        min_idx = np.argmin(dist, axis=-1)
        cluster_idxs[idx] = min_idx     

    return cluster_idxs

rand_points = np.random.randn(100, 2)
clusters = get_mean(rand_points)
plt.scatter(rand_points[:, 0], rand_points[:, 1], c=clusters)

plt.xlabel("X-axis")
plt.ylabel("Y-axis")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## LLM Blocks

In [ ]:
## Decoder layer

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional

# ==================== RoPE ====================

def precompute_freqs_cis(dim: int, end: int = int(32 * 1024), rope_base: float = 1e6,
                         rope_scaling: Optional[dict] = None, device: torch.device = None):
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2, device=device)[: (dim // 2)].float() / dim))
    freqs = freqs.to(device)
    if rope_scaling is not None:
        orig_max = rope_scaling.get("original_max_position_embeddings", 2048)
        factor = rope_scaling.get("factor", 4)
        beta_fast = rope_scaling.get("beta_fast", 4.0)
        beta_slow = rope_scaling.get("beta_slow", 1.0)
        if end / orig_max > 1.0:
            corr_dim = next((i for i in range(dim // 2) if 2 * math.pi / freqs[i] > orig_max), dim // 2)
            power = torch.arange(0, dim // 2, device=freqs.device).float() / max(dim // 2 - 1, 1)
            beta = beta_slow + (beta_fast - beta_slow) * power
            scale = torch.where(
                torch.arange(dim // 2, device=freqs.device) < corr_dim,
                (beta * factor - beta + 1) / (beta * factor),
                1.0 / factor,
            )
            freqs = freqs * scale
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)
    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)
    return freqs_cos, freqs_sin


def apply_rotary_pos_emb(q, k, cos, sin):
    def rotate_half(x):
        return torch.cat((-x[..., x.shape[-1] // 2:], x[..., : x.shape[-1] // 2]), dim=-1)
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


# ==================== RMSNorm ====================

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


# ==================== Attention ====================

class Attention(nn.Module):
    def __init__(self, hidden_dim, num_q_head, num_kv_head, dropout=0.0):
        super().__init__()
        self.num_q_head = num_q_head
        self.num_kv_head = num_kv_head
        self.head_dim = hidden_dim // num_q_head
        self.q_proj = nn.Linear(hidden_dim, num_q_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, num_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, num_kv_head * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_q_head * self.head_dim, hidden_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None, kv_cache=None):
        bsz, in_seq_len, _ = x.shape
        q = self.q_proj(x).reshape(bsz, in_seq_len, self.num_q_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).reshape(bsz, in_seq_len, self.num_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).reshape(bsz, in_seq_len, self.num_kv_head, self.head_dim).transpose(1, 2)

        # 位置偏移 (decode 时从 cache 长度开始)
        start_pos = kv_cache[0].size(2) if kv_cache is not None else 0

        # RoPE: 取对应位置的 freqs
        freqs_cos, freqs_sin = precompute_freqs_cis(
            self.head_dim, start_pos + in_seq_len, device=x.device
        )
        freqs_cos = freqs_cos[start_pos: start_pos + in_seq_len]
        freqs_sin = freqs_sin[start_pos: start_pos + in_seq_len]
        q, k = apply_rotary_pos_emb(q, k, freqs_cos, freqs_sin)

        # KV Cache (RoPE 之后)
        if kv_cache is not None:
            kv_cache[0] = torch.cat([kv_cache[0], k], dim=2)
            kv_cache[1] = torch.cat([kv_cache[1], v], dim=2)
            k, v = kv_cache

        kv_len = k.size(2)

        # GQA: 扩展 k, v 到 num_q_head
        n_rep = self.num_q_head // self.num_kv_head
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)

        # Score
        score = q @ k.transpose(-2, -1) / (self.head_dim ** 0.5)

        # Padding mask
        if attention_mask is not None:
            score = score.masked_fill(attention_mask == 0, float('-inf'))

        # Causal mask (仅 prefill)
        if in_seq_len > 1:
            causal_mask = torch.triu(
                torch.full((in_seq_len, kv_len), float('-inf'), device=x.device),
                diagonal=kv_len - in_seq_len + 1,
            )
            score = score + causal_mask

        attn_weights = torch.softmax(score, dim=-1)
        attn_weights = self.dropout(attn_weights)

        out = (attn_weights @ v).transpose(1, 2).reshape(bsz, in_seq_len, -1)
        return self.o_proj(out), kv_cache


# ==================== SwiGLU FFN ====================

class SwiGLU(nn.Module):
    def __init__(self, hidden_dim, intermediate_dim=None):
        super().__init__()
        if intermediate_dim is None:
            intermediate_dim = ((int(hidden_dim * 8 / 3) + 255) // 256) * 256
        self.gate_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.up_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.down_proj = nn.Linear(intermediate_dim, hidden_dim, bias=False)
    
    def silu(self, x):
        return x * torch.sigmoid(x)

    def forward(self, x):
        return self.down_proj(self.silu(self.gate_proj(x)) * self.up_proj(x))


# ==================== Decoder Layer ====================

class DecoderLayer(nn.Module):
    def __init__(self, hidden_dim, num_q_head, num_kv_head, intermediate_dim=None):
        super().__init__()
        self.attn_norm = RMSNorm(hidden_dim)
        self.attn = Attention(hidden_dim, num_q_head, num_kv_head)
        self.ffn_norm = RMSNorm(hidden_dim)
        self.ffn = SwiGLU(hidden_dim, intermediate_dim)

    def forward(self, x, attention_mask=None, kv_cache=None):
        h = self.attn_norm(x)
        attn_out, kv_cache = self.attn(h, attention_mask, kv_cache)
        x = x + attn_out
        h = self.ffn_norm(x)
        x = x + self.ffn(h)
        return x, kv_cache


# ==================== Decoder ====================

class Decoder(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers, num_q_head, num_kv_head,
                 intermediate_dim=None, max_seq_len=4096):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_kv_head = num_kv_head
        self.head_dim = hidden_dim // num_q_head

        self.embed_tokens = nn.Embedding(vocab_size, hidden_dim)
        self.layers = nn.ModuleList([
            DecoderLayer(hidden_dim, num_q_head, num_kv_head, intermediate_dim)
            for _ in range(num_layers)
        ])
        self.norm = RMSNorm(hidden_dim)
        self.lm_head = nn.Linear(hidden_dim, vocab_size, bias=False)
        self.lm_head.weight = self.embed_tokens.weight  # weight tying

    def init_kv_cache(self, bsz, device=None):
        return [
            [
                torch.zeros(bsz, self.num_kv_head, 0, self.head_dim, device=device),
                torch.zeros(bsz, self.num_kv_head, 0, self.head_dim, device=device),
            ]
            for _ in range(self.num_layers)
        ]

    def forward(self, input_ids, attention_mask=None, kv_caches=None):
        x = self.embed_tokens(input_ids)

        if kv_caches is None:
            kv_caches = [None] * self.num_layers

        new_caches = []
        for i, layer in enumerate(self.layers):
            x, cache = layer(x, attention_mask, kv_caches[i])
            new_caches.append(cache)

        x = self.norm(x)
        return self.lm_head(x), new_caches


# ==================== 测试 ====================

# 超参 (LLaMA-1B 级别)
H, L, Q, KV, V = 2048, 16, 32, 8, 32000
model = Decoder(V, H, L, Q, KV)

# Prefill
ids = torch.randint(0, V, (2, 64))
logits, caches = model(ids)
print(f"prefill logits: {logits.shape}")  # (2, 64, 32000)

# Decode step
kv = model.init_kv_cache(2)
# 先 prefill 填充 cache
_, kv = model(ids[:, :32], kv_caches=kv)
_, kv = model(ids[:, 32:33], kv_caches=kv)
print(f"kv cache len after decode: {kv[0][0].size(2)}")  # 33

prefill logits: torch.Size([2, 64, 32000])
kv cache len after decode: 33


In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from typing import List, Dict

class OnPolicyDistillation:
    def __init__(
        self,
        student_model: nn.Module,          # 学生模型 (当前策略 π_θ)
        teacher_models: List[nn.Module],   # 教师模型列表 (π_Ei)
        teacher_weights: List[float],      # 每个教师对应的权重 w_i
        temperature: float = 1.0,
    ):
        self.student = student_model
        self.teachers = teacher_models
        self.weights = teacher_weights
        self.temperature = temperature
        
        # 确保权重数量与教师数量一致
        assert len(self.teachers) == len(self.weights)

    def _get_logits(
        self, 
        model: nn.Module, 
        input_ids: torch.Tensor, 
        attention_mask: torch.Tensor
    ) -> torch.Tensor:
        """获取模型在当前输入序列上的全词表 logits（带温度调节）"""
        with torch.no_grad() if model is not self.student else torch.enable_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            logits = outputs.logits / self.temperature   # [batch, seq_len, vocab_size]
        return logits

    def compute_opd_loss(
        self,
        on_policy_ids: torch.Tensor,      # 学生模型生成的 on-policy 序列 [B, L]
        attention_mask: torch.Tensor,     # [B, L]
    ) -> torch.Tensor:
        """
        根据已经采样好的 on-policy 轨迹，计算 OPD 损失：
        L = Σ_i w_i * D_KL(π_θ || π_Ei)
        """
        total_loss = 0.0
        
        # 学生 logits（需要梯度）
        student_logits = self._get_logits(self.student, on_policy_ids, attention_mask)
        student_log_probs = F.log_softmax(student_logits, dim=-1)
        student_probs = F.softmax(student_logits, dim=-1)
        
        # 逐教师计算反向 KL 散度
        for teacher, weight in zip(self.teachers, self.weights):
            teacher_logits = self._get_logits(teacher, on_policy_ids, attention_mask)
            teacher_log_probs = F.log_softmax(teacher_logits, dim=-1)
            
            # D_KL( student || teacher ) = Σ_x p_student(x) * (log p_student(x) - log p_teacher(x))
            # 在序列维度求和后求平均
            kl_per_token = torch.sum(
                student_probs * (student_log_probs - teacher_log_probs), dim=-1
            )  # [B, L]
            # 只对有效 token（非 padding）部分计算损失
            masked_kl = kl_per_token * attention_mask
            loss = masked_kl.sum() / attention_mask.sum()
            
            total_loss += weight * loss
            
        return total_loss

    def training_step(
        self,
        prompts: torch.Tensor,           # 输入提示 [B, L_prompt]
        max_new_tokens: int,
        optimizer: torch.optim.Optimizer,
    ) -> float:
        """单步 OPD 训练：采样 -> 计算损失 -> 更新"""
        self.student.eval()  # 采样时学生模型处于 eval 模式
        self.teachers = [t.eval() for t in self.teachers]  # 教师永远 eval
        
        # 1. 从学生策略中采样获得 on-policy 序列
        with torch.no_grad():
            generated = self.student.generate(
                prompts,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=self.temperature,
                pad_token_id=0
            )
        
        # 构建 attention mask（假设 padding 在左侧或右侧，简单处理为全1）
        attention_mask = (generated != 0).float()
        
        self.student.train()  # 学生模型切回训练模式
        
        # 2. 计算 OPD 损失（学生部分需要梯度）
        loss = self.compute_opd_loss(generated, attention_mask)
        
        # 3. 反向传播与优化器更新
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        return loss.item()


# ----------------- 使用示例 -----------------
# 假设已经定义了 student_model, teacher_list, weights
# opd_trainer = OnPolicyDistillation(student_model, teacher_list, weights, temperature=1.0)
# optimizer = torch.optim.AdamW(student_model.parameters(), lr=1e-5)
#
# for batch_prompts in dataloader:
#     loss = opd_trainer.training_step(batch_prompts, max_new_tokens=128, optimizer=optimizer)
#     print(f"OPD loss: {loss}")

## Others

In [ ]:
import torch
from vllm import LLM, SamplingParams


prompts = [
    "请用中文介绍一下你自己。",
    "什么是机器学习？"
]

model = LLM(model="Qwen/Qwen2.5-7B-Instruct", tensor_parallel_size=1, max_model_len=4096, gpu_memory_utilization=0.9)

outputs = model.generate(prompts, SamplingParams(temperature=0.7, top_p=0.9, max_tokens=512))

for o in outputs:
    print(o.outputs[0].text)

In [29]:
import torch

src = torch.ones((2, 5))
index = torch.tensor([[0, 1, 2, 0], [1, 2, 0, 0]])
out = torch.zeros(3, 5, dtype=src.dtype)
out.scatter_add_(0, index, src)
print(out)

tensor([[1., 0., 1., 2., 0.],
        [1., 1., 0., 0., 0.],
        [0., 1., 1., 0., 0.]])


In [52]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "/Users/liushz/Codes/models/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

inputs = ["hello, "]

inputs = tokenizer(inputs, return_tensors="pt")
# outputs = model(**inputs)
outputs = model.generate(**inputs, max_new_tokens=32, do_sample=False)
output_tokens = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(output_tokens)
# logits = outputs.logits
# next_token_id = torch.argmax(logits[:, -1, :], dim=-1)
# next_token = tokenizer.decode(next_token_id)
# print(f"Next token: {next_token}")


hello,  i am trying to create a simple program that will take in an input of a number and then output the sum of all numbers from 1 to that number.


In [ ]:
def simple_grpo(new_logps, old_logps, ref_logps, attention_mask, rewards, epsilon = 0.2, kl_coeff=0.1):
    ratio = torch.exp(new_logps - old_logps)
    kl_ratio = torch.exp(new_logps - ref_logps)
    # reward 
    token_rewards = rewards.unsqueeze(-1).repeat(1, new_logps.shape[1])
    print(token_rewards.shape, rewards.shape)
    token_adv = (token_rewards - rewards.mean()) / (rewards.std() + 1e-8)
    clamped_ratio = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)
    main_loss = torch.min(ratio * token_adv, clamped_ratio * token_adv)
    k3_kl = kl_coeff * (kl_ratio - 1 - torch.log(kl_ratio))
    return -((main_loss - k3_kl) * attention_mask).sum() / attention_mask.sum()

bsz, seq_len = 8, 16
new_logps, ref_logps, old_logps = torch.randn((bsz, seq_len)), torch.randn((bsz, seq_len)), torch.randn((bsz, seq_len))
attention_mask = torch.ones((bsz, seq_len))
rewards = (torch.rand((bsz)) > 0.5).float()
loss = simple_grpo(new_logps, old_logps, ref_logps, attention_mask, rewards)
print(f"GRPO Loss: {loss.item():.4f}")


torch.Size([8, 16]) torch.Size([8])
GRPO Loss: 0.6991


In [ ]:
import torch

def compute_length_penalty(seq_lengths, max_length, cache_length):
    """
    核心特性 4: 软超长惩罚 (Soft Overlong Punishment)
    实现论文中的三段式长度惩罚函数，用于修正奖励信号。
    
    公式：
        R_length(L) = 0,                      if L <= L_max - L_cache
                    = ((L_max - L_cache) - L) / L_cache, if L_max - L_cache < L <= L_max
                    = -1,                      if L > L_max
    
    参数说明：
        max_length (L_max): 模型的生成最大长度。
        cache_length (L_cache): 缓冲区长度，在接近 L_max 前开始施加惩罚。
    """
    # 计算两个关键长度阈值
    safe_boundary = max_length - cache_length
    
    # 创建一个与 seq_lengths 形状相同的全零张量，用于存放惩罚值
    length_penalty = torch.zeros_like(seq_lengths, dtype=torch.float32)
    
    # 阶段 2: 线性递增惩罚 (L_max - L_cache < L <= L_max)
    # 处于此区间的序列，惩罚值从 0 线性递减至 -1
    warning_zone_mask = (seq_lengths > safe_boundary) & (seq_lengths <= max_length)
    length_penalty[warning_zone_mask] = (safe_boundary - seq_lengths[warning_zone_mask]) / cache_length
    
    # 阶段 3: 最大惩罚 (L > L_max)
    # 严重超长的序列受到最强惩罚
    critical_zone_mask = seq_lengths > max_length
    length_penalty[critical_zone_mask] = -1.0
    
    return length_penalty


def apply_overlong_filter(seq_lengths, max_length):
    """
    核心特性 3: 超长过滤 (Overlong Filtering)
    为严重超长的序列生成掩码，在计算损失时将其排除，以防止噪声奖励干扰模型训练。
    
    参数说明：
        seq_lengths (torch.Tensor): 每个样本的序列长度。
        max_length (int): 用于判断是否超长的最大长度阈值。
    
    返回：
        torch.Tensor: 一个布尔掩码，True 表示需要保留的有效token，False 表示应被过滤的超长token。
    """
    # 只有长度严格大于 max_length 的序列才被视为超长，其所有 token 都会被标记为 False
    is_not_overlong = seq_lengths.unsqueeze(-1) <= max_length
    return is_not_overlong


def dynamic_sampling_mask(rewards, gropu_size):
    """
    核心特性 2: 动态采样 / 组过滤 (Dynamic Sampling / Group Filtering)
    此函数用于识别和过滤掉无意义的样本组（组内所有奖励相同），确保优势函数计算有效。
    
    参数说明：
        rewards (torch.Tensor): 所有样本的奖励，形状 (batch_size, )。
        gropu_size (int): 每个 prompt 下采样的回答数量（组大小）。
    
    返回：
        torch.Tensor: 过滤后的样本索引，只保留来自有效组的样本。
    """
    batch_size = rewards.shape[0]
    num_groups = batch_size // gropu_size
    
    valid_indices = []
    for g_idx in range(num_groups):
        start_idx = g_idx * gropu_size
        end_idx = start_idx + gropu_size
        
        # 获取当前组的奖励
        group_rewards = rewards[start_idx:end_idx]
        
        # 核心过滤逻辑：检查组内奖励的标准差是否大于 0
        if group_rewards.std() > 0:
            # 如果组内奖励有差异，则保留该组所有样本
            valid_indices.extend(range(start_idx, end_idx))
    
    # 如果没有有效样本，则返回空列表，表示整个batch都应被跳过
    if not valid_indices:
        return []
    
    return valid_indices


def dapo_loss(new_logps, old_logps, ref_logps, attention_mask, rewards, seq_lengths,
              epsilon_low=0.2, epsilon_high=0.28, kl_coeff=0.0,
              max_length=2048, cache_length=256,
              gropu_size=16, apply_length_penalty=True, apply_overlong_filtering=True):
    """
    整合了 DAPO 四大核心特性的完整损失函数。
    
    参数说明：
        new_logps: 当前策略下的对数概率，形状 (bs, seq_len)
        old_logps: 旧策略下的对数概率，形状 (bs, seq_len)
        ref_logps: 参考策略(如初始模型)下的对数概率，形状 (bs, seq_len)，用于计算 KL 散度
        attention_mask: 注意力掩码，用于标记非填充 token，形状 (bs, seq_len)
        rewards: 每个样本的序列级奖励，形状 (bs, )
        seq_lengths: 每个样本的实际序列长度，形状 (bs, )，用于计算长度惩罚和过滤
        epsilon_low: Clip-Higher 中的下界，论文推荐值 0.2
        epsilon_high: Clip-Higher 中的上界，论文推荐值 0.28
        kl_coeff: KL 散度惩罚系数，DAPO 论文建议设为 0.0（移除 KL 约束）
        max_length (L_max): 软超长惩罚中的最大长度阈值
        cache_length (L_cache): 软超长惩罚中的缓冲区长度
        gropu_size: 每个 prompt 的生成回答数，用于优势计算和动态采样
        apply_length_penalty: 是否应用软超长惩罚
        apply_overlong_filtering: 是否应用超长过滤
    
    返回：
        loss: 最终的 DAPO 损失值
    """
    
    # --- 步骤 1: 应用动态采样 (组过滤) ---
    # 在计算损失前，先过滤掉奖励无差异的无用样本组
    valid_indices = dynamic_sampling_mask(rewards, gropu_size)
    if not valid_indices:
        # 如果整个mini-batch都是无效样本，返回一个零损失，避免梯度计算错误
        return torch.tensor(0.0, device=new_logps.device, requires_grad=True)
    
    # 根据有效索引，裁剪所有输入张量
    valid_indices = torch.tensor(valid_indices, device=new_logps.device)
    new_logps = new_logps[valid_indices]
    old_logps = old_logps[valid_indices]
    ref_logps = ref_logps[valid_indices]
    attention_mask = attention_mask[valid_indices]
    rewards = rewards[valid_indices]
    seq_lengths = seq_lengths[valid_indices]
    
    # --- 步骤 2: 应用软超长惩罚 (奖励塑形) ---
    # 如果启用，则在原始奖励上加上基于长度的惩罚
    if apply_length_penalty:
        length_penalty = compute_length_penalty(seq_lengths, max_length, cache_length)
        rewards = rewards + length_penalty
    
    # --- 步骤 3: 计算组间优势 (Group Relative Advantage) ---
    # 此步骤与 GRPO 保持一致
    token_rewards = rewards.unsqueeze(-1).repeat(1, new_logps.shape[1])
    token_adv = (token_rewards - rewards.mean()) / (rewards.std() + 1e-8)
    
    # --- 步骤 4: 计算重要性采样比率 (Importance Sampling Ratio) ---
    ratio = torch.exp(new_logps - old_logps)
    
    # --- 步骤 5: 核心特性 1 - Clip-Higher (解耦裁剪) ---
    # 使用不对称的裁剪范围来鼓励模型进行更多探索
    pg_losses1 = -token_adv * ratio
    pg_losses2 = -token_adv * torch.clamp(ratio, 1 - epsilon_low, 1 + epsilon_high)
    
    # 核心操作：对每个token，选取导致更大损失的那个裁剪项
    pg_losses = torch.maximum(pg_losses1, pg_losses2)
    
    # --- 步骤 6: 核心特性 5 (可选) - KL 散度惩罚 ---
    # DAPO 论文认为对于复杂推理任务，KL 约束可能不必要，建议设为 0
    kl_ratio = torch.exp(new_logps - ref_logps)
    # 使用低方差 KL 估计器：k3_kl = kl_ratio - 1 - log(kl_ratio)
    k3_kl = kl_coeff * (kl_ratio - 1 - torch.log(kl_ratio))
    
    # 计算每个 token 的总损失
    token_level_loss = pg_losses + k3_kl
    
    # --- 步骤 7: 核心特性 3 - 应用超长过滤掩码 ---
    if apply_overlong_filtering:
        overlong_mask = apply_overlong_filter(seq_lengths, max_length)
        # 将超长样本的token损失置零，使其不参与后续计算
        attention_mask = attention_mask * overlong_mask.float()
    
    # --- 步骤 8: 核心特性 4 - Token级损失归一化 (Token-level Loss) ---
    # 直接除以所有有效 token 的总数，避免序列长度引入的偏差
    total_valid_tokens = attention_mask.sum()
    # 防止除以零
    if total_valid_tokens == 0:
        return torch.tensor(0.0, device=new_logps.device, requires_grad=True)
    
    loss = (token_level_loss * attention_mask).sum() / total_valid_tokens
    
    return loss

# 模拟训练超参数
bsz, seq_len = 16, 32
group_size = 4  # 每个 prompt 生成 4 个回答
max_length = 24  # 模拟的生成最大长度
cache_length = 4 # 模拟的缓冲区长度

# 模拟输入数据
# 假设这个 batch 只有 1 个 prompt，对应 4 个回答，因此 bsz 需要能被 group_size 整除
prompts_in_batch = bsz // group_size
print(f"Batch 中包含 {prompts_in_batch} 个独立 prompt，共 {bsz} 个回答")

new_logps = torch.randn((bsz, seq_len))
old_logps = torch.randn((bsz, seq_len))
ref_logps = torch.randn((bsz, seq_len))

# 创建注意力掩码：模拟不同长度的序列
attention_mask = torch.ones((bsz, seq_len))
seq_lengths = torch.tensor([
    20,  # 此序列长度正常 (20 <= 24-4=20)，不受惩罚和过滤
    26,  # 此序列进入惩罚区域 (20 < 26 <= 24)，线性惩罚
    28,  # 此序列严重超长 (28 > 24)，受最大惩罚和过滤
    22   # 正常序列
]) 
# 根据模拟的长度，将超长部分的掩码置零
for i, l in enumerate(seq_lengths):
    if l < seq_len:
        attention_mask[i, l:] = 0

# 模拟奖励：确保至少有一个组的奖励有差异，以通过动态采样
# 假设 batch 分为两组，每组 4 个样本
rewards = torch.tensor([0.9, 0.1, 0.8, 0.2,  # 第1组：奖励有差异，[0.9, 0.1, 0.8, 0.2] -> std > 0
                        0.5, 0.5, 0.5, 0.5]) # 第2组：奖励完全相同，[0.5, 0.5, 0.5, 0.5] -> std = 0

# 调用 DAPO 损失函数
loss = dapo_loss(
    new_logps, old_logps, ref_logps, attention_mask, rewards, seq_lengths,
    epsilon_low=0.2,       # 标准下界
    epsilon_high=0.28,     # Core Feature 1: Clip-Higher 上界
    kl_coeff=0.0,          # Core Feature 5: 移除 KL 约束 (设为 0.0)
    max_length=max_length, # Core Feature 4: 软超长惩罚参数
    cache_length=cache_length,
    gropu_size=group_size, # Core Feature 2: 动态采样的filter_groups基础
    apply_length_penalty=True,  # 启用软超长惩罚
    apply_overlong_filtering=True # 启用超长过滤
)

print(f"DAPO 损失 (完整特性): {loss.item():.4f}")



tensor(0.5000)
tensor(0.4399)


In [ ]:
# 定义原函数
def f(x):
    return (x**3 - 8)**2

# 一阶导数
def df(x):
    return 2 * (x**3 - 8) * (3*x**2)

# 二阶导数
def ddf(x):
    return 2 * ((3*x**2)**2 + (x**3 - 8) * 6*x)
# 梯度下降
def gradient_descent(x0, lr=0.0001, eps=1e-5, max_iter=10000):
    x = x0
    for i in range(max_iter):
        grad = df(x)
        # 收敛条件：导数接近0
        if abs(x**3 - 8) < eps:
            break
        x = x - lr * grad
    return x, f(x)

# 运行
x_min, y_min = gradient_descent(x0=4.0)
print("===== 梯度下降结果 =====")
print(f"极小值点 x = {x_min:.6f}")
print(f"极小值   y = {y_min:.6f}")


# 牛顿法
def newton_method(x0, eps=1e-6, max_iter=100):
    x = x0
    for i in range(max_iter):
        grad = df(x)
        hess = ddf(x)
        if abs(x**3 - 8) < eps:
            break
        # 迭代公式 x = x - f'(x)/f''(x)
        x = x - grad / hess
    return x, f(x)

# 运行
x_min_n, y_min_n = newton_method(x0=4.0)
print("\n===== 牛顿法结果 =====")
print(f"极小值点 x = {x_min_n:.6f}")
print(f"极小值   y = {y_min_n:.6f}")


===== 梯度下降结果 =====
极小值点 x = 2.000001
极小值   y = 0.000000

===== 牛顿法结果 =====
极小值点 x = 2.000000
极小值   y = 0.000000


: 

In [7]:
import numpy as np  

# 稳定版 softmax（减最大值防溢出）
def softmax(z):
    # z: 1D array
    z_max = np.max(z)
    exp_z = np.exp(z - z_max)
    return exp_z / np.sum(exp_z)

# 手撕 Softmax 雅可比矩阵 (Jacobian)
def softmax_jacobian(z):
    y = softmax(z)
    n = len(y)
    # 初始化雅可比矩阵 [n, n]
    jac = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            if i == j:
                jac[i][j] = y[i] * (1 - y[i])
            else:
                jac[i][j] = -y[i] * y[j]
    return jac

# 测试
z = np.array([1.0, 2.0, 3.0])
y = softmax(z)
jac = softmax_jacobian(z)

print("Softmax 输出 y:\n", y)
print("\nSoftmax 雅可比矩阵 ∂y_i/∂z_j:\n", jac)


Softmax 输出 y:
 [0.09003057 0.24472847 0.66524096]

Softmax 雅可比矩阵 ∂y_i/∂z_j:
 [[ 0.08192507 -0.02203304 -0.05989202]
 [-0.02203304  0.18483645 -0.1628034 ]
 [-0.05989202 -0.1628034   0.22269543]]


In [ ]:
import torch
import torch.nn.functional as F


def get_labels(input_ids, prompt_len, pad_token_id=0):
    """
    构建标签张量，非填充位置为输入ID，填充位置为 -100（CrossEntropyLoss 的 ignore_index）
    
    参数:
        input_ids (torch.Tensor): 输入的 token ID 张量，形状 (batch_size, seq_len)
        prompt_len (torch.Tensor): prompt 的长度，前 prompt_len 个 token 不计算损失
        pad_token_id (int): 填充 token 的 ID，默认值为 0
    
    返回:
        torch.Tensor: 标签张量，形状与 input_ids 相同，填充位置为 -100
    """
    positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0) # [1, seq_len]
    prompt_len = prompt_len.unsqueeze(1)  # [batch_size, 1]
    print(positions < prompt_len)
    labels = input_ids.clone()
    labels[positions < prompt_len] = -100
    labels[labels == pad_token_id] = -100
    return labels




import torch.nn as nn
import torch.optim as optim


model = nn.Linear(32, 8)
inputs = torch.randn(128, 32)
targets = torch.randint(0, 8, (128,))

optimizer = optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


def grad_clip(model, max_norm):
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += torch.norm(p.grad).item() ** 2
    total_norm = total_norm ** 0.5
    clip_coef = max_norm / (total_norm + 1e-6)
    if clip_coef < 1:
        for p in model.parameters():
            if p.grad is not None:
                p.grad.data.mul_(clip_coef)


for epoch in range(10):
    # model.train()
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    grad_clip(model, max_norm=1.0)
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")
    
    
    

Epoch 1, Loss: 2.2247
Epoch 2, Loss: 2.2178
Epoch 3, Loss: 2.2110
Epoch 4, Loss: 2.2041
Epoch 5, Loss: 2.1973
Epoch 6, Loss: 2.1906
Epoch 7, Loss: 2.1838
Epoch 8, Loss: 2.1772
Epoch 9, Loss: 2.1705
Epoch 10, Loss: 2.1639
